In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dados/raw/application_train.csv')

print(df.shape)

In [ ]:
df['DAYS_EMPLOYED'].describe()

In [ ]:
(df['DAYS_EMPLOYED'] == 365243).sum()

In [ ]:
(df['DAYS_EMPLOYED'] == 365243).mean() * 100

Próximo passo
Antes de substituir por NaN, remover ou transformar qualquer coisa, vamos responder uma pergunta:

Os clientes com DAYS_EMPLOYED = 365243 se comportam diferente dos demais?

In [ ]:
df.groupby(df['DAYS_EMPLOYED'] == 365243)['TARGET'].mean() * 100

In [ ]:
df[['EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3']].isnull().mean() * 100

#### Clientes sem EXT_SOURCE_1 possuem comportamento de inadimplência diferente?

In [ ]:
df.groupby(
    df['EXT_SOURCE_1'].isnull()
)['TARGET'].mean() * 100

In [ ]:
high_missing = [
    'COMMONAREA_AVG',
    'LIVINGAPARTMENTS_AVG',
    'NONLIVINGAPARTMENTS_AVG'
]

(df[high_missing].isnull().mean() * 100).sort_values(ascending=False)

In [ ]:
for col in high_missing:
    print(f"\n{col}")
    print(
        df.groupby(df[col].isnull())['TARGET']
          .mean() * 100
    )

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Quantidade de variáveis categóricas: {len(categorical_cols)}')

categorical_cols

In [ ]:
for col in categorical_cols:
    print(f'\n{col}')
    print(df[col].value_counts(dropna=False).head(10))

### ABT V1

In [ ]:
abt_features = [
    'TARGET',

    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',

    'DAYS_BIRTH',
    'DAYS_EMPLOYED',

    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS',

    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',

    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',

    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',

    'REGION_POPULATION_RELATIVE',
    'REGION_RATING_CLIENT',
    'REGION_RATING_CLIENT_W_CITY',

    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]

abt_v1 = df[abt_features].copy()

print(abt_v1.shape)

In [ ]:
abt_v1.isnull().mean().sort_values(ascending=False) * 100

In [ ]:
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]

df[bureau_cols].describe().T

In [ ]:
abt_v1_clean = abt_v1.copy()

In [ ]:
abt_v1_clean.head()

In [ ]:

abt_v1_clean['OCCUPATION_TYPE'] = (
    abt_v1_clean['OCCUPATION_TYPE']
    .fillna('UNKNOWN')
)


In [ ]:
abt_v1_clean

In [ ]:
bureau_cols = [
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]

abt_v1_clean[bureau_cols] = (
    abt_v1_clean[bureau_cols]
    .fillna(0)
)

Criar flags:

In [ ]:
abt_v1_clean['EXT_SOURCE_1_MISSING'] = (
    abt_v1_clean['EXT_SOURCE_1']
    .isnull()
    .astype(int)
)

abt_v1_clean['EXT_SOURCE_3_MISSING'] = (
    abt_v1_clean['EXT_SOURCE_3']
    .isnull()
    .astype(int)
)

In [ ]:
abt_v1_clean.isnull().mean().sort_values(ascending=False) * 100

In [ ]:
numeric_cols = [
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    'AMT_GOODS_PRICE',
    'AMT_ANNUITY',
    'CNT_FAM_MEMBERS'
]

for col in numeric_cols:
    abt_v1_clean[col] = (
        abt_v1_clean[col]
        .fillna(abt_v1_clean[col].median())
    )

In [ ]:
abt_v1_clean.isnull().sum().sum()

In [ ]:
abt_v1_clean.info()

In [ ]:
abt_v1_clean.select_dtypes(include='object').columns.tolist()

In [ ]:
for col in abt_v1_clean.select_dtypes(include='object').columns:
    print(f'{col}: {abt_v1_clean[col].nunique()}')

In [ ]:
abt_v1_clean.shape

In [ ]:
abt_v1_clean.head()

## Pre Modeling 

In [ ]:
X = abt_v1_clean.drop(columns=['TARGET'])
y = abt_v1_clean['TARGET']

print(X.shape)
print(y.shape)

In [ ]:
X.select_dtypes(include='object').columns.tolist()

In [ ]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

In [ ]:
print(X_encoded.shape)


In [ ]:
X_encoded.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(
    max_iter=1000,
    random_state=42
)

modelo.fit(X_train, y_train)

In [ ]:
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

classification_report(y_test, y_pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    xticklabels=['Adimplente', 'Inadimplente'],
    yticklabels=['Adimplente', 'Inadimplente']
)

plt.title('Matriz de Confusão - Logistic Regression')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.show()

In [ ]:

from sklearn.linear_model import LogisticRegression

modelo_balanced = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

modelo_balanced.fit(X_train, y_train)


In [ ]:
y_pred_bal = modelo_balanced.predict(X_test)

y_proba_bal = modelo_balanced.predict_proba(X_test)[:, 1]

In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred_bal))
print("Precision:", precision_score(y_test, y_pred_bal))
print("Recall   :", recall_score(y_test, y_pred_bal))
print("F1-Score :", f1_score(y_test, y_pred_bal))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba_bal))

In [ ]:
confusion_matrix(y_test, y_pred_bal)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_bal = confusion_matrix(y_test, y_pred_bal)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_bal,
    annot=True,
    fmt='d',
    cmap='Greens',
    cbar=False,
    xticklabels=['Adimplente', 'Inadimplente'],
    yticklabels=['Adimplente', 'Inadimplente']
)

plt.title('Matriz de Confusão - Logistic Regression (Balanced)')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.show()

## xgboost

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

y_proba_xgb = xgb_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1-Score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred_xgb)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6, 4))

sns.heatmap(
    cm_xgb,
    annot=True,
    fmt='d',
    cmap='Oranges',
    cbar=False,
    xticklabels=['Adimplente', 'Inadimplente'],
    yticklabels=['Adimplente', 'Inadimplente']
)

plt.title('Matriz de Confusão - XGBoost')
plt.xlabel('Previsto')
plt.ylabel('Real')

plt.show()

In [ ]:
negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()

scale_pos_weight = negativos / positivos

print(scale_pos_weight)

In [ ]:
from xgboost import XGBClassifier

xgb_balanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_balanced.fit(X_train, y_train)

In [ ]:
y_pred_xgb_bal = xgb_balanced.predict(X_test)

y_proba_xgb_bal = xgb_balanced.predict_proba(X_test)[:, 1]

In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred_xgb_bal))
print("Precision:", precision_score(y_test, y_pred_xgb_bal))
print("Recall   :", recall_score(y_test, y_pred_xgb_bal))
print("F1-Score :", f1_score(y_test, y_pred_xgb_bal))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba_xgb_bal))

In [ ]:
import joblib

joblib.dump(
    xgb_balanced,
    '../Model/artifacts/xgb_balanced_model.joblib'
)